In [104]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split

In [105]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sujan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sujan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [106]:
df = pd.read_json("../data/News_Category_Dataset.json", lines=True)
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [107]:
print("Shape:", df.shape)
print()

df.info()

Shape: (209527, 6)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [108]:
df['text'] = df['headline'] + " " + df['short_description']

In [109]:
df = df[['text', 'category']]
df.head()

,text,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [110]:
df['category'].value_counts()

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [111]:
categories = [
'POLITICS',
    'ENTERTAINMENT',
    'SPORTS',
    'WELLNESS',
    'TRAVEL',
    'SCIENCE',
    'TECH'   
]

df = df[df['category'].isin(categories)]

df['category'].value_counts()

category
POLITICS         35602
WELLNESS         17945
ENTERTAINMENT    17362
TRAVEL            9900
SPORTS            5077
SCIENCE           2206
TECH              2104
Name: count, dtype: int64

In [112]:
label_map = {
    'POLITICS':       'Politics',
    'ENTERTAINMENT':  'Entertainment',
    'SPORTS':         'Sports',
    'WELLNESS':       'Wellness',
    'TRAVEL':         'Travel',
    'SCIENCE':        'Science & Tech',
    'TECH':           'Science & Tech',

}

df['category'] = df['category'].map(label_map)

df.head()

,text,category
13,Twitch Bans Gambling Sites After Streamer Scam...,Science & Tech
17,"Maury Wills, Base-Stealing Shortstop For Dodge...",Sports
20,Golden Globes Returning To NBC In January Afte...,Entertainment
21,Biden Says U.S. Forces Would Defend Taiwan If ...,Politics
24,‘Beautiful And Sad At The Same Time’: Ukrainia...,Politics


In [113]:
df.isnull().sum()

text        0
category    0
dtype: int64

In [114]:
df.dropna(inplace=True)

print(df.shape)

(90196, 2)


In [115]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [116]:
def clean_text(text):

    #lowercase
    text = text.lower()

    #remove numbers and symbols
    text = re.sub(r'[^a-z\s]', '',text)

    #split in to words
    words = text.split()

    #remove stop words
    words = [word for word in words
             if word not in stop_words]

    #lemmatization
    words = [lemmatizer.lemmatize(word)
              for word in words]
        
    return " ".join(words)

In [117]:
df['clean_headline'] = df['text'].apply(clean_text)

df.head()

,text,category,clean_headline
13,Twitch Bans Gambling Sites After Streamer Scam...,Science & Tech,twitch ban gambling site streamer scam folk on...
17,"Maury Wills, Base-Stealing Shortstop For Dodge...",Sports,maury will basestealing shortstop dodger dy ma...
20,Golden Globes Returning To NBC In January Afte...,Entertainment,golden globe returning nbc january year offair...
21,Biden Says U.S. Forces Would Defend Taiwan If ...,Politics,biden say u force would defend taiwan china in...
24,‘Beautiful And Sad At The Same Time’: Ukrainia...,Politics,beautiful sad time ukrainian cultural festival...


In [118]:
X = df['clean_headline']
y = df['category']

In [119]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [120]:
print("Training samples:" , len(X_train))
print("Testing sample:", len(X_test))

Training samples: 72156
Testing sample: 18040


In [121]:
for i in range(5):
    print("Original:")
    print(df['text'].iloc[i])

    print("Cleaned:")
    print(df['clean_headline'].iloc[i])


Original:
Twitch Bans Gambling Sites After Streamer Scams Folks Out Of $200,000 One man's claims that he scammed people on the platform caused several popular streamers to consider a Twitch boycott.
Cleaned:
twitch ban gambling site streamer scam folk one man claim scammed people platform caused several popular streamer consider twitch boycott
Original:
Maury Wills, Base-Stealing Shortstop For Dodgers, Dies At 89 Maury Wills, who helped the Los Angeles Dodgers win three World Series titles with his base-stealing prowess, has died.
Cleaned:
maury will basestealing shortstop dodger dy maury will helped los angeles dodger win three world series title basestealing prowess died
Original:
Golden Globes Returning To NBC In January After Year Off-Air For the past 18 months, Hollywood has effectively boycotted the Globes after reports that the HFPA’s 87 members of non-American journalists included no Black members.
Cleaned:
golden globe returning nbc january year offair past month hollywood eff

In [122]:
train_df = pd.DataFrame({
    "headline": X_train,
    "category": y_train
})

test_df = pd.DataFrame({
    "headline": X_test,
    "category": y_test
})

train_df = train_df[train_df['headline'].str.strip() != ""]
test_df = test_df[test_df['headline'].str.strip() != ""]

train_df = train_df.dropna(subset=['headline', 'category']).reset_index(drop=True)
test_df = test_df.dropna(subset=['headline', 'category']).reset_index(drop=True)

print("Train nulls:", train_df.isnull().sum().sum())
print("Test nulls:", test_df.isnull().sum().sum())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.to_csv("../data/train.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

Train nulls: 0
Test nulls: 0
Train shape: (72155, 2)
Test shape: (18039, 2)


In [123]:
# Check for empty strings AFTER cleaning — dropna won't catch these
print("Empty strings in train:", (train_df['headline'] == "").sum())
print("Empty strings in test:", (test_df['headline'] == "").sum())

Empty strings in train: 0
Empty strings in test: 0
